# ROSP-8: Field star ellipticity

PTH Tier 1 metric **ROSP-8** - {{ params.start_dayobs }} to {{ params.end_dayobs }} (dayObs, inclusive)

- Jira epic: [ROSP-8](https://rubinobs.atlassian.net/browse/ROSP-8)
- Confluence metric page: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1939996687](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1939996687)
- Performance Tracking Hub: [https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696](https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/1906802696)

| | |
|---|---|
| Steward | Elana Urbach |
| Owner | Sandrine Thomas |
| Category | 2 - Data Production |
| Layer | 2 |

Method version: `rosp-08-v1.0` (draft until the tag is cut; the frozen commit SHA is recorded on the Confluence page). Changes to the method go through the PTH Method Change Process: https://rubinobs.atlassian.net/wiki/spaces/ROSP/pages/2032959501

**Method.** The ellipticity of a PSF is computed from the second moments of the PSF model (in pixel²):

$$ e_1 = \frac{I_{xx} - I_{yy}}{I_{xx} + I_{yy}}, \qquad e_2 = \frac{2\,I_{xy}}{I_{xx} + I_{yy}}, \qquad e = \sqrt{e_1^2 + e_2^2}. $$

This is the distortion definition of ellipticity used by DESC; its magnitude equals the SRD's
$(\sigma_\mathrm{maj}^2 - \sigma_\mathrm{min}^2)/(\sigma_\mathrm{maj}^2 + \sigma_\mathrm{min}^2)$ (LPM-17 section 3.3.3.3). For every LSSTCam science visit of the
selected programs it is used in two ways:

- $|e|$, **the ellipticity of the visit, is the median over its science detectors of the per-detector $e$.** Each
  detector has its own PSF moments in `cdb_lsstcam.ccdvisit1_quicklook` (`psf_ixx`, `psf_ixy`, `psf_iyy`: the PSF model
  at the centre of the detector), which give one $e$ per detector. The median is over the science detectors that Rapid
  Analysis processed for the visit (detectors 0-188; the corner wavefront sensors and guiders are not used). The
  magnitude is taken per detector, before the median, so an ellipticity pattern that varies across the field of view
  counts in full, as it does in the median over the stars of the field that the SRD specifies. The field is sampled
  once per detector rather than once per star, so the variation within a detector is not included.
- $e_1$ and $e_2$ **are the signed components of the field as a whole.** They come from the median over the detectors
  of each moment (`cdb_lsstcam.visit1_quicklook`: `psf_ixx_median`, `psf_ixy_median`, `psf_iyy_median`). A pattern that
  varies across the field largely cancels in these medians, so $e_1$ and $e_2$ show the ellipticity that the whole
  field has in common, and their magnitude $\sqrt{e_1^2 + e_2^2}$ (the ellipticity of the median moments) is usually
  smaller than $|e|$; the aggregate table gives both. The moments are in DVCS, the pixel frame of the camera, so $e_1$ and
  $e_2$ are fixed to the focal plane, not to the sky: a term that is constant with the rotator angle is in the camera
  or the optics, one that rotates with it comes from the sky side (atmosphere, wind, tracking).

Everything is given per band and for all bands together: the median with the 25th and 75th percentiles over the visits
(table), night by night (median with 25-75 % whiskers), as histograms, and as binned medians (again with 25-75 % whiskers) against telescope elevation, the
physical rotator angle (`visit1_quicklook.physical_rotator_angle`) and the delivered PSF FWHM (uncorrected, as in ROSP-2). Visits need the three median moments,
the ellipticity of at least one science detector, at least
{{ params.min_psf_stars }} PSF stars (`n_psf_star_total`, the `min_psf_stars` parameter), an airmass and a band;
`filter_change_close_loop` and `twilight_near_sun` visits are dropped.

The headline value is the all-band median over the visits of $|e|$.

In [ ]:
# Parameters cell. Set defaults here - gets replaced by the sidecar on Times Square.
# Defaults are the PTH baseline epoch and the standard science blocks.
start_dayobs = 20260629  # First dayObs (inclusive)
end_dayobs = 20261231  # Last dayObs (inclusive)
science_programs = "407, 408, 416, 417, 419, 421"  # bare numbers become BLOCK-<n>; empty = all programs
min_psf_stars = (
    50  # drop visits with fewer PSF stars (n_psf_star_total) than this; 0 = no cut
)
use_cache = (
    False  # cache the ConsDB pull as parquet under ./cache (ignored on Times Square)
)

not_times_square = True

In [ ]:
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# lsst.summit.utils imports spectractor (through lsst.atmospec), which calls matplotlib.use("agg"). Imported after
# the magic below (as ConsDb does in-cluster) it silently drops every later figure, so import it first.
try:
    import lsst.summit.utils  # noqa: F401
except ImportError:
    pass
%matplotlib inline
pd.set_option("display.width", 160)
NOTEBOOK_DIR = os.getcwd()
try:
    not_times_square
except NameError:  # Times Square replaced the parameters cell, which defines the flag
    not_times_square = False

# Bumped through the PTH Method Change Process; the baseline commit is tagged rosp-08-vX.Y.
METHOD_VERSION = "rosp-08-v1.0"

EXCLUDE_REASONS = ("filter_change_close_loop", "twilight_near_sun")
SIGMA_TO_FWHM = 2.0 * np.sqrt(2.0 * np.log(2.0))
PLATESCALE = 0.2  # arcsec / pixel, LSSTCam
# e1, e2: median over the detectors of each moment (visit1_quicklook)
MOMENT_COLUMNS = ("psf_ixx_median", "psf_ixy_median", "psf_iyy_median")
# |e|: the moments of each detector (ccdvisit1_quicklook), science detectors only
DETECTOR_MOMENT_COLUMNS = ("psf_ixx", "psf_ixy", "psf_iyy")
N_SCIENCE_DETECTORS = (
    189  # LSSTCam detectors 0-188; 189-204 are the corner guiders and wavefront sensors
)
ELLIPTICITIES = (("e1", "e1"), ("e2", "e2"), ("e", "|e|"))  # (column, label)
# the aggregate table also gives the ellipticity of the median moments, i.e. the magnitude of (e1, e2)
TABLE_ELLIPTICITIES = ELLIPTICITIES + (("e_median_moments", "sqrt(e1²+e2²)"),)
BANDS = ("u", "g", "r", "i", "z", "y")
BAND_COLORS = dict(
    zip(BANDS, ("#4a3aa7", "#2a78d6", "#1baf7a", "#eda100", "#eb6834", "#e34948"))
)
INK, INK_MUTED, GRID, SURFACE = "#0b0b0b", "#898781", "#e1e0d9", "#fcfcfb"
plt.rcParams.update(
    {
        "figure.facecolor": SURFACE,
        "axes.facecolor": SURFACE,
        "axes.edgecolor": "#c3c2b7",
        "axes.grid": True,
        "grid.color": GRID,
        "grid.linewidth": 0.8,
        "axes.axisbelow": True,
        "legend.frameon": False,
        "font.size": 11,
        "axes.titlesize": 12,
        "savefig.dpi": 150,
    }
)


def parse_day_obs(value) -> int:
    s = str(value).strip().replace("-", "")
    if not re.fullmatch(r"\d{8}", s):
        raise ValueError(f"Not a day_obs: {value!r}")
    return int(s)


def parse_programs(value) -> list:
    """Comma/space separated block names; bare numbers become BLOCK-<n>; empty means no program cut."""
    items = [v for v in re.split(r"[,\s]+", str(value or "").strip()) if v]
    return [f"BLOCK-{it}" if re.fullmatch(r"T?\d+", it) else it for it in items]


def parse_bool(value) -> bool:
    return str(value).strip().lower() in ("true", "1", "yes")


def day_obs_chunks(day_obs_min, day_obs_max, chunk_days=31):
    lo, end = pd.Timestamp(str(day_obs_min)), pd.Timestamp(str(day_obs_max))
    while lo <= end:
        hi = min(lo + pd.Timedelta(days=chunk_days - 1), end)
        yield int(lo.strftime("%Y%m%d")), int(hi.strftime("%Y%m%d"))
        lo = hi + pd.Timedelta(days=1)


def cached(cache_path, loader, use_cache=True) -> pd.DataFrame:
    """Run loader() unless cache_path already holds its parquet output."""
    if cache_path and use_cache and os.path.exists(cache_path):
        df = pd.read_parquet(cache_path)
        print(f"Loaded {len(df)} rows from cache {cache_path}")
        return df
    df = loader()
    if cache_path and len(df):
        os.makedirs(os.path.dirname(os.path.abspath(cache_path)), exist_ok=True)
        df.to_parquet(cache_path, index=False)
    return df


def ellipticity(ixx, ixy, iyy):
    """(e1, e2, |e|) from second moments; NaN where the trace is not positive."""
    ixx, ixy, iyy = (np.asarray(v, dtype=float) for v in (ixx, ixy, iyy))
    with np.errstate(divide="ignore", invalid="ignore"):
        trace = ixx + iyy
        trace = np.where(trace > 0, trace, np.nan)
        e1 = (ixx - iyy) / trace
        e2 = 2.0 * ixy / trace
    return e1, e2, np.hypot(e1, e2)

## ConsDB access

`visit1` (pointing, band) joined to `visit1_quicklook` (PSF moments, PSF sigma, star count, rotator angle) and `exposure`
(`observation_reason`), one row per science visit, chunked by calendar month, through `lsst.summit.utils.ConsDbClient`
from the kernel image (the in-cluster USDF endpoint, or `LSST_CONSDB_PQ_URL` when set). Where that endpoint does not
resolve (e.g. an SDF node) the pull goes through the `rubin_nights` ConsDB client instead, which needs an RSP token.

A second query gives $|e|$: `ccdvisit1_quicklook` (the PSF moments of each detector) joined to `ccdvisit1` (its visit and
detector number) for the same visits, science detectors only. The ellipticity of each detector and its median over the
detectors of the visit are computed in the database (`percentile_cont`), so one row per visit comes back (the median and
the number of detectors in it) instead of one row per detector.

In [ ]:
USDF_CONSDB_URL = (
    "http://consdb-pq.consdb:8080/consdb"  # in-cluster (Times Square, USDF Nublado)
)
VISIT_COLUMNS = (
    "visit_id",
    "day_obs",
    "seq_num",
    "band",
    "physical_filter",
    "science_program",
    "img_type",
    "exp_time",
    "exp_midpt",
    "airmass",
    "zenith_distance",
    "altitude",
    "azimuth",
)
QUICKLOOK_COLUMNS = MOMENT_COLUMNS + (
    "psf_sigma_median",
    "n_psf_star_total",
    "physical_rotator_angle",
)
EXPOSURE_COLUMNS = ("observation_reason",)


def _host_resolves(url):
    import socket
    from urllib.parse import urlparse

    try:
        socket.getaddrinfo(urlparse(url).hostname, None)
        return True
    except (socket.gaierror, UnicodeError, TypeError):
        return False


class ConsDb:
    """ConsDB with .query(sql) -> DataFrame.

    lsst.summit.utils.ConsDbClient when its endpoint resolves (in-cluster on Times Square and USDF Nublado, or
    LSST_CONSDB_PQ_URL). Elsewhere, e.g. an SDF node where consdb-pq.consdb does not resolve, the rubin_nights
    FastAPI client, which needs an RSP token (~/.lsst/usdf_rsp or ACCESS_TOKEN).
    """

    def __init__(self, url=None):
        url = url or os.environ.get("LSST_CONSDB_PQ_URL") or USDF_CONSDB_URL
        if _host_resolves(url):
            from lsst.summit.utils import ConsDbClient

            os.environ["no_proxy"] = os.environ.get("no_proxy", "") + ",.consdb"
            self._client = ConsDbClient(url)
            self.url, self.backend = url, "lsst.summit.utils.ConsDbClient"
        else:
            from rubin_nights import connections

            self._client = connections.get_clients()["consdb"]
            self.url, self.backend = self._client.base_url, "rubin_nights ConsDbFastAPI"

    def query(self, sql):
        result = self._client.query(sql)
        return result if isinstance(result, pd.DataFrame) else result.to_pandas()


def get_consdb_client():
    client = ConsDb()
    print(f"ConsDB client: {client.backend} at {client.url}")
    return client


def _sql_list(values):
    return ", ".join("'" + str(v).replace("'", "''") + "'" for v in values)


def visit_conditions(day_obs_min, day_obs_max, programs, img_types):
    """WHERE conditions on visit1 (alias v), shared by the two queries."""
    where = [f"v.day_obs >= {int(day_obs_min)}", f"v.day_obs <= {int(day_obs_max)}"]
    if programs:
        where.append(f"v.science_program IN ({_sql_list(programs)})")
    if img_types:
        where.append(f"v.img_type IN ({_sql_list(img_types)})")
    return where


def build_query(
    day_obs_min, day_obs_max, programs, instrument="lsstcam", img_types=("science",)
):
    schema = f"cdb_{instrument}"
    where = visit_conditions(day_obs_min, day_obs_max, programs, img_types)
    return (
        "SELECT\n    "
        + ",\n    ".join(
            [f"v.{c}" for c in VISIT_COLUMNS]
            + [f"q.{c}" for c in QUICKLOOK_COLUMNS]
            + [f"e.{c}" for c in EXPOSURE_COLUMNS]
        )
        + f"\nFROM {schema}.visit1 AS v"
        + f"\nLEFT JOIN {schema}.visit1_quicklook AS q ON q.visit_id = v.visit_id"
        + f"\nLEFT JOIN {schema}.exposure AS e ON e.exposure_id = v.visit_id"
        + "\nWHERE "
        + "\n  AND ".join(where)
        + "\nORDER BY v.day_obs, v.seq_num"
    )


def fetch_visits(client, day_obs_min, day_obs_max, programs, chunk_days=31):
    frames = []
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_query(lo, hi, programs))
        required = set(VISIT_COLUMNS + QUICKLOOK_COLUMNS + EXPOSURE_COLUMNS)
        if not required.issubset(df.columns):
            raise RuntimeError(
                f"ConsDB returned an invalid schema for {lo}-{hi}; refusing partial results"
            )
        print(f"  consdb {lo}-{hi}: {len(df):6d} visits")
        if len(df):
            frames.append(df)
    cols = list(VISIT_COLUMNS) + list(QUICKLOOK_COLUMNS) + list(EXPOSURE_COLUMNS)
    if not frames:
        return pd.DataFrame(columns=cols)
    out = pd.concat(frames, ignore_index=True)
    for c in out.columns:
        if (
            c
            not in (
                "band",
                "physical_filter",
                "science_program",
                "observation_reason",
                "img_type",
                "exp_midpt",
            )
            and out[c].dtype == object
        ):
            out[c] = pd.to_numeric(out[c], errors="coerce")
    return out.sort_values(["day_obs", "seq_num"]).reset_index(drop=True)


DETECTOR_COLUMNS = ("visit_id", "n_detectors", "e_detector_median")


def build_detector_query(
    day_obs_min, day_obs_max, programs, instrument="lsstcam", img_types=("science",)
):
    """One row per visit: the median over its science detectors of the per-detector |e|, and the number of detectors.

    |e| of a detector is sqrt((Ixx - Iyy)^2 + 4 Ixy^2) / (Ixx + Iyy) of its PSF model moments. It is computed and
    aggregated in the database, so that one row per visit comes back instead of one per detector. A detector without
    usable moments drops out: Ixx + Iyy <= 0 by the inner condition, NULL or NaN moments by `d.e <= 1` (a NULL never
    passes a comparison and PostgreSQL orders NaN above every number, while a real |e| is at most 1).
    """
    schema = f"cdb_{instrument}"
    ixx, ixy, iyy = (f"q.{c}" for c in DETECTOR_MOMENT_COLUMNS)
    where = visit_conditions(day_obs_min, day_obs_max, programs, img_types) + [
        f"cv.detector < {N_SCIENCE_DETECTORS}",
        f"{ixx} + {iyy} > 0",
    ]
    return (
        "SELECT\n    d.visit_id,\n    COUNT(*) AS n_detectors,"
        + "\n    percentile_cont(0.5) WITHIN GROUP (ORDER BY d.e) AS e_detector_median"
        + "\nFROM (\n    SELECT\n        cv.visit_id,"
        + f"\n        sqrt(({ixx} - {iyy}) * ({ixx} - {iyy}) + 4.0 * {ixy} * {ixy})"
        + f" / NULLIF({ixx} + {iyy}, 0) AS e"
        + f"\n    FROM {schema}.visit1 AS v"
        + f"\n    JOIN {schema}.ccdvisit1 AS cv ON cv.visit_id = v.visit_id"
        + f"\n    JOIN {schema}.ccdvisit1_quicklook AS q ON q.ccdvisit_id = cv.ccdvisit_id"
        + "\n    WHERE "
        + "\n      AND ".join(where)
        + "\n) AS d\nWHERE d.e <= 1\nGROUP BY d.visit_id"
    )


def fetch_detector_ellipticity(
    client, day_obs_min, day_obs_max, programs, chunk_days=7
):
    """Per-visit median of the per-detector |e| (DETECTOR_COLUMNS), in short day_obs chunks to keep each query light."""
    frames = []
    n_queries = 0
    for lo, hi in day_obs_chunks(day_obs_min, day_obs_max, chunk_days):
        df = client.query(build_detector_query(lo, hi, programs))
        n_queries += 1
        if not set(DETECTOR_COLUMNS).issubset(df.columns):
            raise RuntimeError(
                f"ConsDB returned an invalid schema for the per-detector query {lo}-{hi}; refusing partial results"
            )
        if len(df):
            frames.append(df[list(DETECTOR_COLUMNS)])
    if not frames:
        out = pd.DataFrame({c: pd.Series(dtype="float64") for c in DETECTOR_COLUMNS})
        return out.astype({"visit_id": "int64"})
    out = pd.concat(frames, ignore_index=True)
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    print(f"  consdb per-detector |e|: {len(out):6d} visits in {n_queries} queries")
    return out

## Data, ellipticity and selection

The two ConsDB pulls, then per visit $e_1$ and $e_2$ from the three median-moment columns and $|e|$ from the per-detector
query, then the selection table (same cuts as ROSP-2, with "has the median PSF moments" in place of "has the PSF FWHM";
a visit also needs the ellipticity of at least one science detector). How many detectors enter the median of a visit
depends on what Rapid Analysis processed for it: all 189 science detectors by default, fewer when a sparser pattern of
detectors was selected or when detectors have no PSF model. The numbers are printed under the selection.

In [ ]:
start_dayobs, end_dayobs = parse_day_obs(start_dayobs), parse_day_obs(end_dayobs)
if start_dayobs > end_dayobs:
    raise ValueError(
        f"start_dayobs ({start_dayobs}) must be <= end_dayobs ({end_dayobs})"
    )
programs = parse_programs(science_programs)
use_cache = parse_bool(use_cache)
min_psf_stars = int(min_psf_stars)
if min_psf_stars < 0:
    raise ValueError(f"min_psf_stars ({min_psf_stars}) must be >= 0")
cache_dir = (
    os.path.join(NOTEBOOK_DIR, "cache") if (use_cache and not_times_square) else None
)
cache_tag = f"v1_{start_dayobs}_{end_dayobs}_{'_'.join(p.replace('BLOCK-', '') for p in programs) or 'all'}"
print(
    f"dayObs {start_dayobs} to {end_dayobs}; programs {programs or 'all'}; "
    f"min_psf_stars {min_psf_stars}"
)

consdb = get_consdb_client()
raw = cached(
    cache_dir and os.path.join(cache_dir, f"visits_{cache_tag}.parquet"),
    lambda: fetch_visits(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)
detector_e = cached(
    cache_dir and os.path.join(cache_dir, f"detector_e_{cache_tag}.parquet"),
    lambda: fetch_detector_ellipticity(consdb, start_dayobs, end_dayobs, programs),
    use_cache,
)

visits = raw.copy()
visits["band"] = visits["band"].astype("string").str.strip().str[0]
# e1, e2: the field as a whole, from the median moments (their magnitude is kept for comparison only)
visits["e1"], visits["e2"], visits["e_median_moments"] = ellipticity(
    *(visits[c] for c in MOMENT_COLUMNS)
)
# |e| of the visit: the median over its science detectors of the per-detector |e|
by_visit = detector_e.set_index("visit_id")
visits["e"] = visits["visit_id"].map(by_visit["e_detector_median"]).astype(float)
visits["n_detectors"] = visits["visit_id"].map(by_visit["n_detectors"]).astype(float)
visits["rotator"] = visits["physical_rotator_angle"].astype(float)
visits["fwhm"] = visits["psf_sigma_median"].astype(float) * SIGMA_TO_FWHM * PLATESCALE
visits["time"] = pd.to_datetime(
    visits["exp_midpt"], utc=True, errors="coerce", format="ISO8601"
).dt.tz_localize(None)

steps = [
    (
        "has the median PSF moments (Ixx + Iyy > 0)",
        visits["e_median_moments"].notna(),
    ),
    (
        "has the per-detector |e| (at least one science detector)",
        visits["e"].notna(),
    ),
    (
        f"n_psf_star_total >= {min_psf_stars}",
        visits["n_psf_star_total"].astype(float) >= min_psf_stars,
    ),
    (
        "has airmass >= 1 and a known band",
        (visits["airmass"] >= 1) & visits["band"].isin(BANDS),
    ),
    (
        f"observation_reason not in {EXCLUDE_REASONS}",
        ~visits["observation_reason"].isin(EXCLUDE_REASONS),
    ),
]
keep = pd.Series(True, index=visits.index)
rows = []
for name, mask in steps:
    mask = mask.fillna(False)
    rows.append((name, int((keep & ~mask).sum()), int((keep & mask).sum())))
    keep &= mask
print(f"\nStart: {len(visits)} visits")
print(
    pd.DataFrame(rows, columns=["cut", "removed", "remaining"]).to_string(index=False)
)
sel = visits[keep].copy()
print(f"\n{len(sel)} visits on {sel['day_obs'].nunique()} nights")
print(f"visits with a rotator angle: {int(sel['rotator'].notna().sum())} of {len(sel)}")
if len(sel):
    n_det = sel["n_detectors"]
    print(
        f"science detectors in the median |e| of a visit: median {n_det.median():.0f}, "
        f"range {n_det.min():.0f} to {n_det.max():.0f}; "
        f"{int((n_det < N_SCIENCE_DETECTORS / 2).sum())} visits with fewer than half of the {N_SCIENCE_DETECTORS}"
    )
print(
    pd.crosstab(sel["science_program"], sel["band"], margins=True).reindex(
        columns=[*BANDS, "All"], fill_value=0
    )
)

## Results

### Aggregate table

Median $e_1$, $e_2$ and $|e|$ with the 25th and 75th percentiles over the visits, per band and for all bands. $|e|$ is
the per-visit median of the per-detector ellipticity (the headline quantity). The last three columns, `sqrt(e1²+e2²)`,
give the ellipticity of the median moments of the visit, i.e. the magnitude of its $e_1$ and $e_2$, for comparison: what
$|e|$ has in excess of it is ellipticity that varies across the field and cancels in the median moments.

In [ ]:
def quartiles(x):
    x = pd.to_numeric(x, errors="coerce").dropna()
    return (
        pd.Series(
            {"p25": x.quantile(0.25), "median": x.median(), "p75": x.quantile(0.75)}
        )
        if len(x)
        else pd.Series({"p25": np.nan, "median": np.nan, "p75": np.nan})
    )


def aggregate_table(df):
    groups = [("all", df)] + [
        (b, df[df["band"] == b]) for b in BANDS if (df["band"] == b).any()
    ]
    rows = []
    for name, g in groups:
        row = {
            "band": name,
            "n_visits": len(g),
            "n_nights": g["day_obs"].nunique(),
            "median_airmass": g["airmass"].median(),
        }
        for col, label in TABLE_ELLIPTICITIES:
            q = quartiles(g[col])
            row.update(
                {
                    f"{label} p25": q["p25"],
                    f"{label} median": q["median"],
                    f"{label} p75": q["p75"],
                }
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index("band")


summary = aggregate_table(sel)
print(
    f'headline: all-band median |e| = {summary.loc["all", "|e| median"]:.4f} '
    f'[{summary.loc["all", "|e| p25"]:.4f}, {summary.loc["all", "|e| p75"]:.4f}]  '
    f'(n = {int(summary.loc["all", "n_visits"])} visits on {int(summary.loc["all", "n_nights"])} nights)'
)
moments_label = TABLE_ELLIPTICITIES[-1][1]
print(
    f"for comparison, the ellipticity of the median moments: all-band median {moments_label} = "
    f'{summary.loc["all", f"{moments_label} median"]:.4f}'
)
summary.round(4)

### Nightly time series

$|e|$: all bands together (top row), then one panel per band, the nightly median with 25-75 % whiskers. Below it the
nightly medians of $e_1$ and $e_2$ per band on one axis each, so a sign change or a band offset is visible directly.

In [ ]:
def nightly_quartiles(df, col):
    g = df.groupby("day_obs")[col]
    out = pd.DataFrame(
        {
            "p25": g.quantile(0.25),
            "median": g.median(),
            "p75": g.quantile(0.75),
            "n": g.size(),
        }
    )
    out["date"] = pd.to_datetime(out.index.astype(str))
    return out


def whiskers(ax, nightly, color, label, offset_days=0.0, ms=4):
    x = nightly["date"] + pd.Timedelta(days=offset_days)
    ax.errorbar(
        x,
        nightly["median"],
        yerr=np.array(
            [nightly["median"] - nightly["p25"], nightly["p75"] - nightly["median"]]
        ),
        fmt="o",
        ms=ms,
        color=color,
        ecolor=color,
        elinewidth=1.0,
        capsize=0,
        alpha=0.9,
        label=label,
    )


bands_present = [b for b in BANDS if (sel["band"] == b).any()]
panels = [("all", "all bands", INK)] + [
    (b, f"{b} band", BAND_COLORS[b]) for b in bands_present
]
fig, axes = plt.subplots(
    len(panels),
    1,
    figsize=(13, 2.9 * len(panels) + 1),
    sharex=True,
    sharey=True,
    squeeze=False,
    layout="constrained",
)
for ax, (band, title, color) in zip(axes[:, 0], panels):
    sub = sel if band == "all" else sel[sel["band"] == band]
    whiskers(ax, nightly_quartiles(sub, "e"), color, "|e|")
    ax.set_ylabel("|e|")
    ax.set_title(
        f"{title}  (n = {len(sub)} visits, {sub['day_obs'].nunique()} nights)",
        loc="left",
        fontsize=11,
    )
    ax.set_xlabel("night (day_obs)")
    ax.tick_params(labelbottom=True)
axes[0, 0].legend(
    fontsize=9, title="marker: nightly median; whisker: 25-75 %", title_fontsize=9
)
fig.align_ylabels(axes[:, 0])
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(13, 7.5), sharex=True, layout="constrained")
for ax, (col, label) in zip(axes, ELLIPTICITIES[:2]):
    for band in bands_present:
        nightly = nightly_quartiles(sel[sel["band"] == band], col)
        ax.plot(
            nightly["date"],
            nightly["median"],
            marker="o",
            ms=4,
            lw=1.0,
            color=BAND_COLORS[band],
            label=f"{band} band",
        )
    nightly = nightly_quartiles(sel, col)
    ax.plot(
        nightly["date"],
        nightly["median"],
        marker="o",
        ms=4,
        lw=1.6,
        color=INK,
        label="all bands",
    )
    ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_ylabel(f"nightly median {label}")
    ax.set_xlabel("night (day_obs)")
    ax.tick_params(labelbottom=True)
axes[0].legend(ncol=4, fontsize=9)
fig.align_ylabels(axes)
plt.show()

### Histograms

$e_1$, $e_2$ and $|e|$ with all bands overlaid (fraction of that band's visits per bin), the median (tick) and 25 / 75 %
(dotted) of each band marked above the curves and the all-band median as the vertical line. The legend quotes the three values.

In [ ]:
def band_histograms(ax, df, col, bins, fmt="{:+.3f}"):
    """Per-band step histograms of df[col] with quartile ticks above and the all-band median line."""
    ymax = 0
    for band in bands_present:
        d = df.loc[df["band"] == band, col].dropna()
        if not len(d):
            continue
        q = quartiles(d)
        counts, _, _ = ax.hist(
            d.clip(bins[0], bins[-1]),
            bins=bins,
            histtype="step",
            lw=2,
            color=BAND_COLORS[band],
            weights=np.full(len(d), 1 / len(d)),
            label=f"{band}: {fmt.format(q['median'])} [{fmt.format(q['p25'])}, {fmt.format(q['p75'])}]  (n = {len(d)})",
        )
        ymax = max(ymax, counts.max())
    for k, band in enumerate(bands_present):
        q = quartiles(df.loc[df["band"] == band, col])
        y = ymax * (1.06 + 0.05 * k)
        ax.plot([q["p25"], q["p75"]], [y, y], color=BAND_COLORS[band], lw=1.2, ls=":")
        ax.plot([q["median"]], [y], marker="|", ms=10, mew=2, color=BAND_COLORS[band])
    q = quartiles(df[col])
    ax.axvline(
        q["median"],
        color=INK,
        lw=1.2,
        label=f"all bands: {fmt.format(q['median'])} [{fmt.format(q['p25'])}, {fmt.format(q['p75'])}]",
    )
    ax.axvline(q["p25"], color=INK, lw=0.8, ls=":")
    ax.axvline(q["p75"], color=INK, lw=0.8, ls=":")
    ax.set_ylim(0, ymax * (1.06 + 0.05 * len(bands_present) + 0.04) if ymax else 1)
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylabel("fraction of the band's visits per bin")
    ax.legend(fontsize=8.5, title="median [25 %, 75 %]", title_fontsize=9)


e_max = float(np.nanpercentile(sel["e"], 99.5)) if len(sel) else 0.2
signed_bins = np.arange(-e_max, e_max + 0.0025, 0.005)
abs_bins = np.arange(0.0, e_max + 0.0025, 0.005)
fig, axes = plt.subplots(1, 3, figsize=(20, 5.8))
for ax, (col, label) in zip(axes, ELLIPTICITIES):
    band_histograms(ax, sel, col, abs_bins if col == "e" else signed_bins)
    if col != "e":
        ax.axvline(0, color=INK_MUTED, lw=1, ls="--")
    ax.set_xlabel(label)
    ax.set_title(label)
plt.show()

### Against elevation, rotator angle and delivered FWHM

Binned medians of $e_1$, $e_2$ and $|e|$ in bins of telescope elevation, of the physical rotator angle, and of the
delivered PSF FWHM (`psf_sigma_median` × 2√(2 ln 2) × 0.2″/pixel, uncorrected, as in ROSP-2), all bands (black) and per
band, with the visits per bin underneath. The marker is the median of the visits in the bin and the whisker runs from
their 25th to their 75th percentile: it shows how the visits in the bin are spread, not how well the median is known. The all-band point sits at the centre of its bin and the bands
are drawn next to it, on both sides, so that the whiskers do not hide each other: every point between two bin limits
(those of the histogram underneath) belongs to that bin. Bins with fewer than 10 visits are skipped.

An elevation trend points at gravity-driven flexure, the airmass or the differential chromatic refraction; a
$\cos 2\theta$ / $\sin 2\theta$ pattern in $e_1$ / $e_2$ against the rotator angle is an ellipticity fixed to the telescope
(alt-az) seen through the rotating camera (`physical_rotator_angle` is RotTelPos), a flat offset is camera-fixed; a
trend with the FWHM separates an ellipticity that is a fixed fraction of the PSF size (optics, tracking) from a fixed
absolute contribution that the atmosphere dilutes in poor seeing.

In [ ]:
MIN_VISITS_PER_BIN = 10


def binned_median(x, y, edges, min_count=MIN_VISITS_PER_BIN):
    """Median and 25th / 75th percentiles of y in bins of x, with the count; sparse bins are left NaN."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    idx = np.digitize(x[ok], edges) - 1
    rows = []
    for i in range(len(edges) - 1):
        d = y[ok][idx == i]
        if len(d) >= min_count:
            p25, med, p75 = np.percentile(d, [25, 50, 75])
        else:
            p25 = med = p75 = np.nan
        rows.append(
            {
                "center": 0.5 * (edges[i] + edges[i + 1]),
                "width": edges[i + 1] - edges[i],
                "n": len(d),
                "median": med,
                "p25": p25,
                "p75": p75,
            }
        )
    return pd.DataFrame(rows)


def bin_shifts(n_series, span=0.7):
    """Sideways offsets, in bin widths, for n_series drawn in one bin.

    The first series stays at the bin centre and the others are spread on both sides of it, never on top of it.
    """
    n_side = n_series - 1
    slots = np.linspace(-0.5 * span, 0.5 * span, n_side + 1 + n_side % 2)
    return [0.0, *np.delete(slots, len(slots) // 2)[:n_side]]


def plot_binned(ax, st, color, label, main=False, shift=0.0):
    """One series of binned_median(): the medians joined by a line, with whiskers from the 25th to the 75th percentile.

    The whisker is the spread of the visits in the bin, not the uncertainty of the median. shift (in bin widths, see
    bin_shifts) moves the series sideways so that the whiskers of several series do not hide each other. The whiskers
    of the series that are not the main one are drawn lighter than their medians.
    """
    series = ax.errorbar(
        st["center"] + shift * st["width"],
        st["median"],
        yerr=np.array([st["median"] - st["p25"], st["p75"] - st["median"]]),
        marker="o" if main else "s",
        ms=5 if main else 3.5,
        lw=2 if main else 1.2,
        elinewidth=1.5 if main else 1.0,
        capsize=0,
        color=color,
        label=label,
        zorder=3 if main else 2.5,
    )
    if not main:
        for whisker in series[2]:
            whisker.set_alpha(0.5)


def binned_panels(df, xcol, edges, xlabel, groups, columns=ELLIPTICITIES):
    """One column per (column, label) in columns: binned medians (top) and visits per bin (bottom) against df[xcol].

    Marker: median of the visits in the bin; whisker: their 25th to 75th percentile. The first of groups is drawn at
    the bin centres and the others next to it, on both sides (bin_shifts).
    """
    fig, axes = plt.subplots(
        2,
        len(columns),
        figsize=(6.2 * len(columns), 8),
        sharex=True,
        gridspec_kw={"height_ratios": [3, 1]},
        squeeze=False,
        layout="constrained",
    )
    for k, (col, label) in enumerate(columns):
        ax, ax_n = axes[0, k], axes[1, k]
        for (name, color, g), shift in zip(groups, bin_shifts(len(groups))):
            st = binned_median(g[xcol], g[col], edges)
            if st["median"].notna().any():
                series = "all bands" if name == "all" else f"{name} band"
                plot_binned(ax, st, color, series, main=name == "all", shift=shift)
            d = g.loc[g[col].notna(), xcol].dropna()
            if len(d):
                ax_n.hist(
                    d,
                    bins=edges,
                    histtype="step",
                    lw=2 if name == "all" else 1.2,
                    color=color,
                )
        if col != "e":
            ax.axhline(0, color=INK_MUTED, lw=1, ls="--")
        ax.set_title(f"{label}  (n = {int(df[col].notna().sum())})", fontsize=11)
        ax.set_ylabel(f"binned median {label}")
        ax.set_xlabel(xlabel)
        ax.tick_params(labelbottom=True)
        ax_n.set_xlabel(xlabel)
        ax_n.set_ylabel("visits")
        ax_n.set_xlim(edges[0], edges[-1])
    handles, names = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles,
        names,
        loc="outside upper center",
        ncol=len(handles),
        fontsize=9,
        title="marker: median of the visits in the bin; whisker: their 25th to 75th percentile",
        title_fontsize=9,
    )
    fig.align_ylabels(axes[:, 0])
    return fig


groups = [("all", INK, sel)] + [
    (b, BAND_COLORS[b], sel[sel["band"] == b]) for b in bands_present
]
binned_panels(
    sel, "altitude", np.arange(20.0, 90.1, 5.0), "telescope elevation (deg)", groups
)
plt.show()

rot = sel["rotator"].dropna()
if len(rot):
    lo = np.floor(rot.min() / 15.0) * 15.0
    hi = np.ceil(rot.max() / 15.0) * 15.0
    binned_panels(
        sel,
        "rotator",
        np.arange(lo, hi + 1.0, 15.0),
        "physical rotator angle (deg)",
        groups,
    )
    plt.show()
else:
    print(
        "no rotator angle in the selection: visit1_quicklook.physical_rotator_angle is empty"
    )

fwhm = sel["fwhm"].dropna()
if len(fwhm):
    lo = np.floor(fwhm.quantile(0.005) * 10) / 10
    hi = np.ceil(fwhm.quantile(0.995) * 10) / 10
    binned_panels(
        sel,
        "fwhm",
        np.arange(lo, hi + 0.05, 0.1),
        "PSF FWHM as delivered (arcsec)",
        groups,
    )
    plt.show()

### Elevation and rotator angle together

Binned medians of $e_1$, $e_2$ and $|e|$ on the elevation × rotator-angle plane (all bands; same bins as above), with the
visits per cell on the right. Cells with fewer than 10 visits are blank. The two axes separate the camera-fixed and the
telescope-fixed terms: a term fixed to the camera is flat along the rotator axis, one fixed to the telescope (alt-az:
gravity, the drives, refraction) runs along it as $\cos 2\theta$ / $\sin 2\theta$, with an amplitude that can change with
elevation.

In [ ]:
def binned_median_2d(
    df, xcol, ycol, col, x_edges, y_edges, min_count=MIN_VISITS_PER_BIN
):
    """Median of df[col] on the (x, y) grid and the counts; cells below min_count are NaN."""
    x = df[xcol].to_numpy(dtype=float)
    y = df[ycol].to_numpy(dtype=float)
    v = df[col].to_numpy(dtype=float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(v)
    ix = np.digitize(x[ok], x_edges) - 1
    iy = np.digitize(y[ok], y_edges) - 1
    med = np.full((len(y_edges) - 1, len(x_edges) - 1), np.nan)
    n = np.zeros_like(med, dtype=int)
    for i in range(len(x_edges) - 1):
        for j in range(len(y_edges) - 1):
            d = v[ok][(ix == i) & (iy == j)]
            n[j, i] = len(d)
            if len(d) >= min_count:
                med[j, i] = np.median(d)
    return med, n


alt_edges = np.arange(20.0, 90.1, 5.0)
rot = sel["rotator"].dropna()
if len(rot):
    rot_edges = np.arange(
        np.floor(rot.min() / 15.0) * 15.0, np.ceil(rot.max() / 15.0) * 15.0 + 1.0, 15.0
    )
    fig, axes = plt.subplots(1, 4, figsize=(24, 5.8), layout="constrained")
    for ax, (col, label) in zip(axes[:3], ELLIPTICITIES):
        med, n = binned_median_2d(sel, "altitude", "rotator", col, alt_edges, rot_edges)
        if col == "e":
            kw = dict(
                cmap="viridis",
                vmin=0,
                vmax=np.nanmax(med) if np.isfinite(med).any() else 1,
            )
        else:
            lim = np.nanmax(np.abs(med)) if np.isfinite(med).any() else 1
            kw = dict(cmap="RdBu_r", vmin=-lim, vmax=lim)
        im = ax.pcolormesh(alt_edges, rot_edges, np.ma.masked_invalid(med), **kw)
        fig.colorbar(im, ax=ax, label=f"binned median {label}")
        ax.set_title(f"{label}  (all bands, n = {int(n.sum())})", fontsize=11)
        ax.set_xlabel("telescope elevation (deg)")
        ax.set_ylabel("physical rotator angle (deg)")
    im = axes[3].pcolormesh(
        alt_edges, rot_edges, np.ma.masked_equal(n, 0), cmap="Blues"
    )
    fig.colorbar(im, ax=axes[3], label="visits")
    axes[3].set_title("visits per cell", fontsize=11)
    axes[3].set_xlabel("telescope elevation (deg)")
    axes[3].set_ylabel("physical rotator angle (deg)")
    plt.show()
else:
    print(
        "no rotator angle in the selection: visit1_quicklook.physical_rotator_angle is empty"
    )

### Elevation dependence in the alt-az frame

The $e_1$ and $e_2$ above are in the camera pixel frame (DVCS), which turns with the rotator, so a term that is fixed
to the telescope (gravity, the azimuth and elevation drives, differential chromatic refraction) is shared between the two
components and partly averages out over the rotator angles. Here the ellipticity of each visit is rotated into the alt-az
frame with its physical rotator angle $\theta$ (`physical_rotator_angle`: the mean `MTRotator` position during the
exposure, i.e. RotTelPos), with $x$ along azimuth and $y$ along elevation as seen on the sky:

$$ e_1^\mathrm{AltAz} + i\,e_2^\mathrm{AltAz} = -\,(e_1 + i\,e_2)\,e^{-2i\theta}, \qquad
e_1^\mathrm{AltAz} = -(e_1 \cos 2\theta + e_2 \sin 2\theta), \qquad
e_2^\mathrm{AltAz} = e_1 \sin 2\theta - e_2 \cos 2\theta. $$

The convention is that of [SMTN-019](https://smtn-019.lsst.io) and of the RubinTV PSF az/el plots (`aaRot` in
`lsst.summit.extras.plotting.psfPlotting`): projected on the sky, with azimuth to the right and elevation up,
$+y_\mathrm{DVCS}$ points at the angle $-\theta$ and $+x_\mathrm{DVCS}$ at $-(\theta + 90^\circ)$. At $\theta = 0$ the DVCS $y$
axis lies along azimuth and the $x$ axis along elevation, the camera turns clockwise on the sky as $\theta$ increases,
and the two frames differ by a rotation only (no flip); an ellipticity turns by twice the angle of the axes. The
magnitude $\sqrt{e_1^2 + e_2^2}$ does not change, and neither does $|e|$, which is taken per detector.

Positive $e_1^\mathrm{AltAz}$ is a PSF elongated along azimuth (parallel to the horizon), negative along elevation;
$e_2^\mathrm{AltAz}$ is the elongation along the diagonals (positive: towards increasing azimuth and elevation
together). Differential chromatic refraction always stretches the PSF along elevation, so it should show as a negative
$e_1^\mathrm{AltAz}$ that grows towards low elevation, with nothing in $e_2^\mathrm{AltAz}$: a check on the sign
conventions as much as a result. DCR alone would make it much larger in the blue bands; here the slope against
$\tan z$ is about as large in r, i and z as in g, so most of the elevation term is achromatic (an elongation along
elevation that grows like $\tan z$), with an extra chromatic part in u and g. Same bins and 25-75 % whiskers (the spread of the visits in each bin) as above; visits without a rotator angle are left out.

In [ ]:
ALTAZ_ELLIPTICITIES = (
    ("e1_altaz", "alt-az e1"),
    ("e2_altaz", "alt-az e2"),
)  # (column, label)
ALTAZ_SIGN_NOTES = (
    "> 0: along azimuth, < 0: along elevation",
    "> 0: along the +azimuth, +elevation diagonal",
)


def rotate_to_altaz(e1, e2, rotator_deg):
    """(e1, e2) from the DVCS pixel frame to the alt-az frame (x along azimuth, y along elevation, on the sky).

    Projected on the sky with azimuth to the right and elevation up, +y_DVCS points at -RotTelPos and +x_DVCS at
    -(RotTelPos + 90 deg) (SMTN-019; aaRot in lsst.summit.extras.plotting.psfPlotting). That is a rotation without a
    flip, and a spin-2 ellipticity turns by twice the angle of the axes: e_altaz = -e_dvcs exp(-2i RotTelPos).
    """
    e_dvcs = np.asarray(e1, dtype=float) + 1j * np.asarray(e2, dtype=float)
    theta = np.radians(np.asarray(rotator_deg, dtype=float))
    e_altaz = -e_dvcs * np.exp(-2j * theta)
    return e_altaz.real, e_altaz.imag


sel["e1_altaz"], sel["e2_altaz"] = rotate_to_altaz(sel["e1"], sel["e2"], sel["rotator"])
n_altaz = int(sel["e1_altaz"].notna().sum())
print(
    f"visits rotated into the alt-az frame: {n_altaz} of {len(sel)} (the others have no rotator angle)"
)
if n_altaz:
    fig = binned_panels(
        sel,
        "altitude",
        np.arange(20.0, 90.1, 5.0),
        "telescope elevation (deg)",
        [("all", INK, sel)]
        + [(b, BAND_COLORS[b], sel[sel["band"] == b]) for b in bands_present],
        columns=ALTAZ_ELLIPTICITIES,
    )
    for ax, note in zip(fig.axes, ALTAZ_SIGN_NOTES):  # the two top panels
        ax.set_title(f"{ax.get_title()}\n{note}", fontsize=11)
    plt.show()

### Against the SRD specification

The LSST SRD (LPM-17, section 3.3.3.3, Table 13) specifies, for a delivered seeing of 0.69″ in a field with a zenith
distance of at most 10°, that the ellipticity distribution of unresolved sources has a median not larger than SE1, with
no more than EF1 % of images exceeding the ellipticity SE2, for the r and i bands: SE1 = 0.04 / 0.05 / 0.03, EF1 =
5 / 10 / 5 %, SE2 = 0.07 / 0.1 / 0.05 at the design / minimum / stretch levels. The SRD defines the ellipticity as
$(\sigma_\mathrm{maj}^2 - \sigma_\mathrm{min}^2)/(\sigma_\mathrm{maj}^2 + \sigma_\mathrm{min}^2)$ of the best-fit elliptical
Gaussian, which is the $|e|$ computed here. The closest sample this notebook can form is
the r and i band visits at elevation ≥ `SRD_MIN_ELEVATION` whose delivered FWHM is within ± `SRD_SEEING_TOL` of 0.69″.
The table compares its median $|e|$ with SE1 and its fraction above SE2 with EF1 at each level; the figure shows the
$|e|$ distribution of that sample (left, with SE1 and SE2 marked) and, because few visits land exactly at 0.69″, the
binned median $|e|$ of the r and i band visits at elevation ≥ `SRD_MIN_ELEVATION` against the delivered FWHM
(right, with 25-75 % whiskers for the spread of the visits in each bin, the SE1 levels as lines and the seeing
window shaded).

This is a comparison with the spirit of the requirement, not its verification: the SRD median is over the
unresolved sources in the field of view of one image, whereas $|e|$ here is the median over the science detectors of the
ellipticity of the PSF model at the centre of each detector (one sample per processed detector, not one per star, so the
variation within a detector is not included); the delivered seeing window and elevation cut are choices
(`SRD_SEEING_TOL`, `SRD_MIN_ELEVATION`); and the SRD's PSF is a double Gaussian fit rather than the Rapid Analysis PSF
model. Furthermore, we require at least `SRD_MIN_NIGHTS` to be able to really evaluate this parameter. Fewer nights than that will be noted at the bottom of the output.

In [ ]:
SRD_LEVELS = {  # LPM-17 (rev. 2018-01-30) Table 13, r and i bands: SE1 = max median e, EF1 = max % of images above SE2
    "design": {"SE1": 0.04, "SE2": 0.07, "EF1": 5.0},
    "minimum": {"SE1": 0.05, "SE2": 0.10, "EF1": 10.0},
    "stretch": {"SE1": 0.03, "SE2": 0.05, "EF1": 5.0},
}
SRD_BANDS = ("r", "i")
SRD_SEEING = 0.69  # arcsec, delivered
SRD_SEEING_TOL = 0.05  # arcsec: visits with |FWHM - 0.69| <= this form the sample
SRD_MIN_ELEVATION = 80.0  # deg, i.e. zenith distance <= 10 deg
SRD_MIN_NIGHTS = 10


near_zenith = sel[sel["band"].isin(SRD_BANDS) & (sel["altitude"] >= SRD_MIN_ELEVATION)]
srd = near_zenith[(near_zenith["fwhm"] - SRD_SEEING).abs() <= SRD_SEEING_TOL]
print(
    f"{', '.join(SRD_BANDS)} band, elevation >= {SRD_MIN_ELEVATION:g} deg: {len(near_zenith)} visits; "
    f"of which FWHM within {SRD_SEEING:g} +/- {SRD_SEEING_TOL:g} arcsec: {len(srd)} visits on {srd['day_obs'].nunique()} nights"
)
rows = []
for level, spec in SRD_LEVELS.items():
    median_e = srd["e"].median() if len(srd) else np.nan
    frac_above = 100 * (srd["e"] > spec["SE2"]).mean() if len(srd) else np.nan
    rows.append(
        {
            "level": level,
            "SE1 (max median |e|)": spec["SE1"],
            "median |e|": median_e,
            "median meets SE1": (
                bool(median_e <= spec["SE1"]) if np.isfinite(median_e) else None
            ),
            "SE2 (threshold)": spec["SE2"],
            "EF1 (max % above SE2)": spec["EF1"],
            "% above SE2": frac_above,
            "fraction meets EF1": (
                bool(frac_above <= spec["EF1"]) if np.isfinite(frac_above) else None
            ),
            "n_visits": len(srd),
        }
    )
srd_table = pd.DataFrame(rows).set_index("level")
evaluable = srd["day_obs"].nunique() >= SRD_MIN_NIGHTS


fig, (ax_h, ax_f) = plt.subplots(1, 2, figsize=(17, 5.8), layout="constrained")
level_colors = dict(zip(SRD_LEVELS, ("#2a78d6", "#eb6834", "#1baf7a")))
if len(srd):
    e_bins = np.arange(0.0, max(0.12, float(srd["e"].max()) + 0.01), 0.0025)
    ax_h.hist(srd["e"], bins=e_bins, color=INK_MUTED, alpha=0.35, lw=0)
    ax_h.hist(srd["e"], bins=e_bins, histtype="step", color=INK, lw=1.5)
    ax_h.axvline(
        srd["e"].median(),
        color=INK,
        lw=2,
        label=f'median |e| = {srd["e"].median():.4f}',
    )
    for level, spec in SRD_LEVELS.items():
        ax_h.axvline(
            spec["SE1"],
            color=level_colors[level],
            lw=1.4,
            ls="--",
            label=f'{level}: SE1 = {spec["SE1"]:.2f}',
        )
        ax_h.axvline(
            spec["SE2"],
            color=level_colors[level],
            lw=1.4,
            ls=":",
            label=f'{level}: SE2 = {spec["SE2"]:.2f}',
        )
    ax_h.set_xlim(e_bins[0], e_bins[-1])
    ax_h.legend(
        fontsize=8.5,
        title="dashed: SE1 (median); dotted: SE2 (threshold)",
        title_fontsize=8.5,
    )
ax_h.set_xlabel("|e|")
ax_h.set_ylabel("visits")
ax_h.set_title(
    f"{', '.join(SRD_BANDS)}, elevation >= {SRD_MIN_ELEVATION:g} deg, FWHM = {SRD_SEEING:g} +/- {SRD_SEEING_TOL:g} arcsec  (n = {len(srd)})",
    fontsize=11,
)

fwhm_nz = near_zenith["fwhm"].dropna()
if len(fwhm_nz):
    lo = np.floor(fwhm_nz.quantile(0.005) * 10) / 10
    hi = np.ceil(fwhm_nz.quantile(0.995) * 10) / 10
    edges = np.arange(lo, hi + 0.05, 0.1)
    ax_f.axvspan(
        SRD_SEEING - SRD_SEEING_TOL,
        SRD_SEEING + SRD_SEEING_TOL,
        color=INK_MUTED,
        alpha=0.15,
        lw=0,
        label="seeing window",
    )
    srd_groups = [(f"{', '.join(SRD_BANDS)} together", INK, near_zenith)] + [
        (f"{b} band", BAND_COLORS[b], near_zenith[near_zenith["band"] == b])
        for b in SRD_BANDS
    ]
    for k, (label, color, g) in enumerate(srd_groups):
        st = binned_median(g["fwhm"], g["e"], edges)
        if st["median"].notna().any():
            shift = bin_shifts(len(srd_groups))[k]
            plot_binned(ax_f, st, color, label, main=k == 0, shift=shift)
    for level, spec in SRD_LEVELS.items():
        ax_f.axhline(
            spec["SE1"],
            color=level_colors[level],
            lw=1.4,
            ls="--",
            label=f'{level}: SE1 = {spec["SE1"]:.2f}',
        )
    ax_f.set_xlim(edges[0], edges[-1])
    ax_f.legend(
        fontsize=8.5,
        ncol=2,
        title="marker: median of the bin; whisker: 25-75 % of its visits",
        title_fontsize=8.5,
    )
ax_f.set_xlabel("PSF FWHM as delivered (arcsec)")
ax_f.set_ylabel("binned median |e|")
ax_f.set_title(
    f"{', '.join(SRD_BANDS)}, elevation >= {SRD_MIN_ELEVATION:g} deg  (n = {len(near_zenith)})",
    fontsize=11,
)
plt.show()

srd_table.round(4)

print(
    "median meets SE1 is",
    (
        bool(median_e <= spec["SE1"])
        if evaluable and np.isfinite(median_e)
        else "not evaluable"
    ),
)
print(
    "fraction meets EF1 is",
    (
        bool(frac_above <= spec["EF1"])
        if evaluable and np.isfinite(frac_above)
        else "not evaluable"
    ),
)

## Provenance

What this run was computed with: the method version, the input columns and the ellipticity definition, the kernel image
and the library versions.

In [ ]:
import datetime
import platform
import shutil
import subprocess


def stack_version():
    out = []
    try:
        from lsst.summit.utils import version as su_version

        out.append(f"summit_utils {su_version.__version__}")
    except Exception:
        out.append("summit_utils not importable")
    if shutil.which("eups"):
        res = subprocess.run(
            ["eups", "list", "-s", "lsst_distrib"], capture_output=True, text=True
        )
        out.append(f"lsst_distrib {' '.join(res.stdout.split()) or 'not set up'}")
    return "; ".join(out)


provenance = {
    "method version": METHOD_VERSION,
    "run (UTC)": datetime.datetime.now(datetime.timezone.utc).strftime(
        "%Y-%m-%d %H:%M"
    ),
    "moment columns, |e|": "cdb_lsstcam.ccdvisit1_quicklook."
    + ", ".join(DETECTOR_MOMENT_COLUMNS)
    + f" (Summit Rapid Analysis; PSF model at the centre of each detector, pixel^2; science detectors 0-{N_SCIENCE_DETECTORS - 1}; "
    "producer version not recorded in ConsDB)",
    "moment columns, e1 and e2": "cdb_lsstcam.visit1_quicklook."
    + ", ".join(MOMENT_COLUMNS)
    + " (Summit Rapid Analysis; median over detectors, pixel^2; producer version not recorded in ConsDB)",
    "ellipticity": "e1 = (Ixx - Iyy) / (Ixx + Iyy), e2 = 2 Ixy / (Ixx + Iyy), e = sqrt(e1^2 + e2^2), camera pixel frame; "
    "|e| of a visit = median over its science detectors of the per-detector e; e1, e2 of a visit from the median moments",
    "rotator angle": "cdb_lsstcam.visit1_quicklook.physical_rotator_angle",
    "PSF FWHM": "cdb_lsstcam.visit1_quicklook.psf_sigma_median x 2 sqrt(2 ln 2) x 0.2 arcsec/pixel, as delivered (ROSP-2)",
    "selection": f"img_type science; programs {programs or 'all'}; median PSF moments and the |e| of at least one "
    f"science detector; n_psf_star_total >= {min_psf_stars}; "
    f"airmass >= 1; known band; observation_reason not in {EXCLUDE_REASONS}",
    "ConsDB endpoint": consdb.url,
    "kernel image": os.environ.get("JUPYTER_IMAGE_SPEC", "not a Nublado pod"),
    "image description": os.environ.get("IMAGE_DESCRIPTION", "n/a"),
    "stack": stack_version(),
    "python / numpy / pandas": f"{platform.python_version()} / {np.__version__} / {pd.__version__}",
}
with pd.option_context("display.max_colwidth", None):
    display(pd.DataFrame.from_dict(provenance, orient="index", columns=["value"]))